# AHC015 teacher continuation training

Save & Run All用Notebook。`ppo-20260827-015219`のbest checkpoint（iteration 79）から、seed 15031で5時間継続する。評価は2,048ケースを2 iterationごとに行う。GPU T4 x2、Internet On、`GITHUB_TOKEN`と`WANDB_API_KEY`のSecret accessを有効にして実行する。

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA GPU is required"
assert torch.cuda.device_count() == 2, "Select the Kaggle GPU T4 x2 accelerator"

for index in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(index)
    capability = torch.cuda.get_device_capability(index)
    print(index, name, capability)
    assert "T4" in name and capability == (7, 5), "GPU T4 x2 is required"

test_tensor = torch.zeros((8, 12, 10, 10), device="cuda")
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA tensor:", test_tensor.shape, test_tensor.device)
del test_tensor
torch.cuda.empty_cache()

In [ ]:
import base64
import os
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

repo_dir = Path("/kaggle/working/ahc-ml")
expected_commit = "c899cbec9de8e79b03a6d5d326ffa48298dc49f7"
assert not repo_dir.exists(), f"Clean session required: {repo_dir} already exists"

github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
credentials = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env["GIT_CONFIG_COUNT"] = "1"
git_env["GIT_CONFIG_KEY_0"] = "http.extraHeader"
git_env["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {credentials}"

try:
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "feature/ahc015-teacher",
            "--single-branch",
            "https://github.com/e1jirou/ahc-ml.git",
            str(repo_dir),
        ],
        check=True,
        env=git_env,
    )
finally:
    del github_token
    del credentials
    del git_env

subprocess.run(
    ["git", "-C", str(repo_dir), "checkout", "--detach", expected_commit],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
    text=True,
).strip()
assert actual_commit == expected_commit

os.chdir(repo_dir)
print("Repository commit:", actual_commit)
print("Current directory:", os.getcwd())

In [ ]:
%pip install --quiet torchview==0.2.7

from importlib.metadata import version

print("torchview:", version("torchview"))

In [ ]:
import os
from pathlib import Path

import torch
import wandb
from kaggle_secrets import UserSecretsClient

wandb_api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_api_key
del wandb_api_key
assert wandb.login(verify=True)

api = wandb.Api()
source_run = api.run("eijirou-personal/ahc-ml/7gqyexpf")
assert source_run.name == "ppo-20260827-015219"
assert source_run.state == "finished"
artifact_name = (
    "eijirou-personal/ahc-ml/"
    "ppo-20260827-015219-training-checkpoint:v0"
)
checkpoint_dir = Path("/kaggle/working/checkpoints") / source_run.name
artifact = api.artifact(artifact_name, type="model")
downloaded_dir = Path(artifact.download(root=checkpoint_dir))
checkpoint_path = downloaded_dir / "best-training.pt"
assert checkpoint_path.is_file()

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
assert checkpoint["epoch"] == 79
print("Source W&B run:", source_run.name, source_run.id)
print("Checkpoint:", checkpoint_path)
print("Epoch:", checkpoint["epoch"])
print("Paired gain:", checkpoint["metrics"]["evaluation/paired_gain"])
del checkpoint

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo_dir = Path("/kaggle/working/ahc-ml")
assert checkpoint_path.is_file()

train_env = os.environ.copy()
train_env["PYTHONPATH"] = str(repo_dir / "python")
train_env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

subprocess.run(
    [
        sys.executable,
        "-m",
        "examples.ahc015.python.train",
        "--config",
        "examples/ahc015/config.toml",
        "--seed",
        "15031",
        "--device",
        "cuda",
        "--wandb-mode",
        "online",
        "--max-hours",
        "5",
        "--data-parallel",
        "--rollout-processes",
        "2",
        "--micro-batch-size",
        "512",
        "--evaluation-interval",
        "2",
        "--evaluation-episodes",
        "2048",
        "--resume",
        str(checkpoint_path),
    ],
    cwd=repo_dir,
    env=train_env,
    check=True,
)

run_dirs = sorted((repo_dir / "outputs/ahc015").glob("ppo-*"))
assert run_dirs
latest_run = run_dirs[-1]
assert (latest_run / "best-training.pt").is_file()
assert (latest_run / "last.pt").is_file()
print("5-hour accelerated training completed:", latest_run)